# 기본 RAG (Retrieval-Augmented Generation)

모델은 우리가 가진 건강 팁 문서를 알지 못합니다. 질문과 관련된 문서를 먼저 찾아 함께 넘겨주면 모델은 그 내용을 근거로 답합니다. 이것이 **RAG**(Retrieval-Augmented Generation)입니다. 앞 노트북에서 만든 임베딩을 Azure AI Search에 저장하고 검색과 생성을 하나로 연결해 봅니다.

사용하는 패키지는 세 가지입니다. **`azure-ai-projects`**로 Foundry 프로젝트에 연결하고, **`openai`**로 임베딩과 Chat Completions를 수행하며, **`azure-search-documents`**로 벡터 검색을 합니다. 주제는 건강 & 피트니스 🍏입니다.

**🎯 미션**

1. 건강 팁 문서를 임베딩해 Azure AI Search 벡터 인덱스에 저장합니다.
2. 사용자 질문으로 관련 문서를 검색하고 그 내용을 근거로 답변을 생성합니다.
3. 실습을 마친 뒤 인덱스를 정리합니다.

> **사전 준비**: [README](README.md)의 Azure AI Search 구성 + `.env`의 `SEARCH_*` 변수 3개, 그리고 [02-embeddings.ipynb](02-embeddings.ipynb) 완료.
> **안내문**: 이 노트북은 의료 조언을 제공하지 않습니다.


## 1. 설정

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# 상위 폴더의 .env 로드
load_dotenv(Path().absolute().parent / ".env")

endpoint = os.environ["PROJECT_ENDPOINT"]
chat_model = os.environ["MODEL_NAME"]                    # 배포 이름 (예: gpt-5-mini)
embedding_model = os.environ["TEXT_EMBEDDING_MODEL"]     # 배포 이름 (예: text-embedding-3-small)

# Entra ID(keyless) 인증 — 사전에 `az login` 필요
project = AIProjectClient(endpoint=endpoint, credential=DefaultAzureCredential())

# 프로젝트 범위 클라이언트 (chat/responses/agents 용)
client = project.get_openai_client()

# 임베딩 전용 클라이언트 — 프로젝트 범위 엔드포인트에는 /embeddings 라우트가 없어 리소스 범위를 쓴다
account_endpoint = endpoint.split("/api/projects/")[0]
embedding_client = project.get_openai_client(base_url=f"{account_endpoint}/openai/v1")
print("✅ AIProjectClient / OpenAI client 준비 완료")

# Azure AI Search 클라이언트 (실습 편의상 관리자 키 사용 — 운영에서는 Entra ID 권장)
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient

search_endpoint = os.environ["SEARCH_ENDPOINT"]
search_api_key = os.environ["SEARCH_API_KEY"]
search_index_name = os.environ["SEARCH_INDEX_NAME"]
print("✅ Search 환경 변수 로드 완료")


## 2. 샘플 건강 데이터 생성

실제 시나리오에서는 CSV나 PDF를 읽어 청크로 나누고 임베딩하지만, 여기서는 간단한 문서 6개로 진행합니다.

In [ ]:
health_tips = [
    {"id": "doc1", "content": "Daily 30-minute walks help maintain a healthy weight and reduce stress.", "source": "General Fitness"},
    {"id": "doc2", "content": "Stay hydrated by drinking 8-10 cups of water per day.", "source": "General Fitness"},
    {"id": "doc3", "content": "Consistent sleep patterns (7-9 hours) improve muscle recovery.", "source": "General Fitness"},
    {"id": "doc4", "content": "For cardio endurance, try interval training like HIIT.", "source": "Workout Advice"},
    {"id": "doc5", "content": "Warm up with dynamic stretches before running to reduce injury risk.", "source": "Workout Advice"},
    {"id": "doc6", "content": "Balanced diets typically include protein, whole grains, fruits, vegetables, and healthy fats.", "source": "Nutrition"},
]
print(f"Created {len(health_tips)} health tips.")

## 3. 벡터 인덱스 생성

Azure AI Search에서 벡터 필드 정의에는 `vector_search_profile_name`이 필요하며, 벡터 검색 설정의 프로필 이름과 일치해야 합니다. HNSW 알고리즘 구성을 사용해 인덱스를 생성하는 헬퍼를 정의합니다.

In [ ]:
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    HnswParameters,
    VectorSearchAlgorithmKind,
    VectorSearchAlgorithmMetric,
    VectorSearchProfile,
)

def create_healthtips_index(endpoint: str, api_key: str, index_name: str, dimension: int):
    """벡터 검색이 가능한 health tips 인덱스를 생성(기존 인덱스는 재생성)합니다."""
    index_client = SearchIndexClient(endpoint=endpoint, credential=AzureKeyCredential(api_key))

    try:
        index_client.delete_index(index_name)
        print(f"Deleted existing index: {index_name}")
    except Exception:
        pass  # 아직 없는 경우

    vector_search = VectorSearch(
        algorithms=[
            HnswAlgorithmConfiguration(
                name="myHnsw",
                kind=VectorSearchAlgorithmKind.HNSW,
                parameters=HnswParameters(
                    m=4,
                    ef_construction=400,
                    ef_search=500,
                    metric=VectorSearchAlgorithmMetric.COSINE,
                ),
            )
        ],
        profiles=[VectorSearchProfile(name="myHnswProfile", algorithm_configuration_name="myHnsw")],
    )

    index = SearchIndex(
        name=index_name,
        fields=[
            SimpleField(name="id", type=SearchFieldDataType.String, key=True),
            SearchableField(name="content", type=SearchFieldDataType.String),
            SimpleField(name="source", type=SearchFieldDataType.String, filterable=True),
            SearchField(
                name="embedding",
                type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                searchable=True,
                vector_search_dimensions=dimension,
                vector_search_profile_name="myHnswProfile",
            ),
        ],
        vector_search=vector_search,
    )
    index_client.create_or_update_index(index)
    print(f"✅ Created index: {index_name} (dimension={dimension})")

## 4. 임베딩 생성 및 업로드

1. 샘플 문서 하나를 임베딩해 **차원 수를 확인**하고,
2. 인덱스를 생성한 뒤,
3. 모든 건강 팁을 임베딩과 함께 **업로드**합니다.

In [ ]:
# 1) 임베딩 차원 확인
sample_emb = embedding_client.embeddings.create(model=embedding_model, input=[health_tips[0]["content"]])
embedding_length = len(sample_emb.data[0].embedding)
print(f"✅ Embedding dimension: {embedding_length}")

# 2) 인덱스 생성
create_healthtips_index(search_endpoint, search_api_key, search_index_name, dimension=embedding_length)

# 3) 문서 임베딩 + 업로드
search_client = SearchClient(
    endpoint=search_endpoint,
    index_name=search_index_name,
    credential=AzureKeyCredential(search_api_key),
)

search_docs = []
for doc in health_tips:
    emb = embedding_client.embeddings.create(model=embedding_model, input=[doc["content"]])
    search_docs.append({**doc, "embedding": emb.data[0].embedding})

result = search_client.upload_documents(documents=search_docs)
print(f"✅ Uploaded {len(search_docs)} documents.")


## 5. 기본 RAG 흐름

### 5.1 검색(Retrieve)
사용자 질문을 임베딩 → 벡터 인덱스에서 상위 문서 검색

### 5.2 생성(Generate)
검색된 문서를 컨텍스트로 채팅 모델에 전달해 응답 생성

> 실제 시나리오에서는 더 정교한 청크 분할·하이브리드 검색·리랭킹 전략을 사용합니다.

In [ ]:
from azure.search.documents.models import VectorizedQuery

def rag_chat(query: str, top_k: int = 3) -> str:
    # 1) 사용자 질문 임베딩
    user_vec = embedding_client.embeddings.create(model=embedding_model, input=[query]).data[0].embedding

    # 2) 벡터 검색
    vector_query = VectorizedQuery(vector=user_vec, k_nearest_neighbors=top_k, fields="embedding")
    results = search_client.search(
        search_text="",
        vector_queries=[vector_query],
        select=["content", "source"],
    )
    top_docs = [f"Source: {r['source']} => {r['content']}" for r in results]

    # 3) 검색 결과를 컨텍스트로 응답 생성
    system_text = (
        "You are a health & fitness assistant.\n"
        "Answer user questions using ONLY the text from these docs.\n"
        "Docs:\n" + "\n".join(top_docs) + "\nIf unsure, say 'I'm not sure'.\n"
    )
    response = client.chat.completions.create(
        model=chat_model,
        messages=[
            {"role": "system", "content": system_text},
            {"role": "user", "content": query},
        ],
    )
    return response.choices[0].message.content


## 6. 쿼리 실행해보기

In [ ]:
user_query = "What's a good short cardio routine for me if I'm busy?"
answer = rag_chat(user_query)
print("🗣️ User Query:", user_query)
print("🤖 RAG Answer:", answer)

## 7. 정리 (Clean-up)

실습을 마쳤다면 아래 셀로 인덱스를 삭제할 수 있습니다. AI Search 리소스 자체는 Azure Portal에서 삭제하세요.


In [ ]:
# (선택) 인덱스 삭제
# SearchIndexClient(endpoint=search_endpoint, credential=AzureKeyCredential(search_api_key)).delete_index(search_index_name)
# print(f"🗑️ Deleted index: {search_index_name}")

## ✅ 미션 완료

**무엇을 만들었나:**

- ✓ 건강 팁 문서를 임베딩해 저장한 Azure AI Search 벡터 인덱스
- ✓ 질문으로 상위 문서를 검색(Retrieve)하고 그 내용을 근거로 답변을 생성(Generate)하는 RAG 파이프라인
- ✓ 실습 후 인덱스를 삭제하는 정리 절차

실제 시나리오에서는 더 정교한 청크 분할·하이브리드 검색·리랭킹 전략을 얹게 됩니다. 다음 장에서는 이 흐름을 에이전트로 감싸 봅니다 → [05. Agent Service](../05-agent-service/README.md)
